# TSV Evaluation — WFST Phonetisaurus G2P

Trains and evaluates Phonetisaurus models directly from `dicts/train.tsv` and `dicts/test.tsv`.
Metrics: **PER** (Phone Error Rate) and **WER** (Word Error Rate).

> **Run this notebook inside the Docker container** (`./run_phonetisaurus.ps1`).

## Setup

In [1]:
import subprocess
import sys
import os
import pandas as pd
from pathlib import Path

sys.path.append(os.path.abspath("../utils"))
from tsv_metrics import tokenize_ipa, load_tsv, compute_per_wer, collect_errors

In [2]:
current_dir = Path().resolve()
dicts_dir   = current_dir / "dicts"
train_dir   = current_dir / "train"
output_dir  = current_dir / "output"
wlist_dir   = current_dir / "wlist"

train_dir.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)
wlist_dir.mkdir(exist_ok=True)

print("Working directory:", current_dir)

Working directory: /workspaces/Taglog-G2P/notebook


## Data Preparation

Convert `train.tsv` (`word\tIPA`) to Phonetisaurus dict format (`word\tphone1 phone2 ...`).
IPA is tokenized using `tokenize_ipa`, which strips stress/syllable markers and keeps
diacritics, length marks, and tie-bar affricates as single tokens.
Multi-word grapheme entries are skipped — Phonetisaurus operates word-by-word.

In [3]:
tsv_train_dict = dicts_dir / "tsv_train.dict"

written = 0
skipped = 0
with open(dicts_dir / "train.tsv", encoding="utf-8") as fin, \
     open(tsv_train_dict, "w", encoding="utf-8") as fout:
    for line in fin:
        parts = line.rstrip("\n").split("\t")
        if len(parts) < 2:
            continue
        word, ipa = parts[0].strip(), parts[1].strip()
        if " " in word:
            skipped += 1
            continue
        phones = tokenize_ipa(ipa)
        if phones:
            fout.write(f"{word}\t{' '.join(phones)}\n")
            written += 1

print(f"Entries written   : {written}")
print(f"Multi-word skipped: {skipped}")

Entries written   : 18433
Multi-word skipped: 1270


In [4]:
ref = load_tsv(dicts_dir / "test.tsv", skip_multiword=True)

tsv_test_wlist = wlist_dir / "tsv_test.wlist"
with open(tsv_test_wlist, "w", encoding="utf-8") as f:
    for word in ref:
        f.write(word + "\n")

print(f"Test words (single-word): {len(ref)}")

Test words (single-word): 2312


## Baseline Model

Default Phonetisaurus settings with `--seq2_del`, mirroring the baseline from `main.ipynb`
but trained and evaluated on the TSV data.

In [5]:
baseline_model = train_dir / "tsv_baseline"

result = subprocess.run(
    [
        "phonetisaurus-train",
        "--lexicon", str(tsv_train_dict),
        "--seq2_del",
        "--model", str(baseline_model),
    ],
    capture_output=True, text=True
)
print(result.stderr)

INFO:phonetisaurus-train:2026-06-10 04:29:25:  Checking command configuration...
INFO:phonetisaurus-train:2026-06-10 04:29:25:  Checking lexicon for reserved characters: '}', '|', '_'...
INFO:phonetisaurus-train:2026-06-10 04:29:25:  Aligning lexicon...
INFO:phonetisaurus-train:2026-06-10 04:29:47:  Training joint ngram model...
INFO:phonetisaurus-train:2026-06-10 04:29:51:  Converting ARPA format joint n-gram model to WFST format...
INFO:phonetisaurus-train:2026-06-10 04:29:53:  G2P training succeeded: /workspaces/Taglog-G2P/notebook/train/tsv_baseline.fst



In [6]:
baseline_output = output_dir / "tsv_baseline_output.txt"

with open(baseline_output, "w", encoding="utf-8") as f:
    subprocess.run(
        [
            "phonetisaurus-apply",
            "--model", str(baseline_model) + ".fst",
            "--word_list", str(tsv_test_wlist),
        ],
        stdout=f, text=True
    )

hyp_baseline = {}
with open(baseline_output, encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            hyp_baseline[parts[0]] = parts[1].split()

print(f"Predictions generated: {len(hyp_baseline)}")

Predictions generated: 2312


In [7]:
metrics_baseline = compute_per_wer(ref, hyp_baseline)

print("=== Baseline Model (train.tsv + --seq2_del) ===")
print(f"  PER : {metrics_baseline['per']:.4f}  ({metrics_baseline['per']*100:.2f}%)")
print(f"  WER : {metrics_baseline['wer']:.4f}  ({metrics_baseline['wer']*100:.2f}%)")
print(f"  Words evaluated  : {metrics_baseline['n_words']}")
print(f"  Correct words    : {metrics_baseline['n_correct']}")
print(f"  Total ref phones : {metrics_baseline['n_ref_phones']}")
print(f"  Total edit dist  : {metrics_baseline['n_edit']}")

=== Baseline Model (train.tsv + --seq2_del) ===
  PER : 0.0740  (7.40%)
  WER : 0.3724  (37.24%)
  Words evaluated  : 2312
  Correct words    : 1451
  Total ref phones : 18086
  Total edit dist  : 1339


## Tuned Model

Best hyperparameter configuration from the grid search in `main.ipynb`
(saved in `output/hyperparam_results.csv`):

| Param | Value |
|---|---|
| `ngram_order` | 4 |
| `seq1_max` | 1 |
| `seq2_max` | 2 |
| `seq2_del` | True |
| `grow` | True |

Cross-validation mean PER on `train.dict`: **7.88%**

In [8]:
tuned_model = train_dir / "tsv_tuned"

result = subprocess.run(
    [
        "phonetisaurus-train",
        "--lexicon",     str(tsv_train_dict),
        "--ngram_order", "4",
        "--seq1_max",    "1",
        "--seq2_max",    "2",
        "--seq2_del",
        "--grow",
        "--model",       str(tuned_model),
    ],
    capture_output=True, text=True
)
print(result.stderr)

INFO:phonetisaurus-train:2026-06-10 04:29:54:  Checking command configuration...
INFO:phonetisaurus-train:2026-06-10 04:29:54:  Checking lexicon for reserved characters: '}', '|', '_'...
INFO:phonetisaurus-train:2026-06-10 04:29:54:  Aligning lexicon...
INFO:phonetisaurus-train:2026-06-10 04:30:08:  Training joint ngram model...
INFO:phonetisaurus-train:2026-06-10 04:30:09:  Converting ARPA format joint n-gram model to WFST format...
INFO:phonetisaurus-train:2026-06-10 04:30:09:  G2P training succeeded: /workspaces/Taglog-G2P/notebook/train/tsv_tuned.fst



In [9]:
tuned_output = output_dir / "tsv_tuned_output.txt"

with open(tuned_output, "w", encoding="utf-8") as f:
    subprocess.run(
        [
            "phonetisaurus-apply",
            "--model",     str(tuned_model) + ".fst",
            "--word_list", str(tsv_test_wlist),
        ],
        stdout=f, text=True
    )

hyp_tuned = {}
with open(tuned_output, encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            hyp_tuned[parts[0]] = parts[1].split()

metrics_tuned = compute_per_wer(ref, hyp_tuned)

print("=== Tuned Model (ngram_order=4, seq1_max=1, seq2_max=2, seq2_del, grow) ===")
print(f"  PER : {metrics_tuned['per']:.4f}  ({metrics_tuned['per']*100:.2f}%)")
print(f"  WER : {metrics_tuned['wer']:.4f}  ({metrics_tuned['wer']*100:.2f}%)")
print(f"  Words evaluated  : {metrics_tuned['n_words']}")
print(f"  Correct words    : {metrics_tuned['n_correct']}")
print(f"  Total ref phones : {metrics_tuned['n_ref_phones']}")
print(f"  Total edit dist  : {metrics_tuned['n_edit']}")

=== Tuned Model (ngram_order=4, seq1_max=1, seq2_max=2, seq2_del, grow) ===
  PER : 0.0813  (8.13%)
  WER : 0.4109  (41.09%)
  Words evaluated  : 2312
  Correct words    : 1362
  Total ref phones : 18086
  Total edit dist  : 1470


## Results Summary

In [10]:
summary = pd.DataFrame([
    {"Model": "Baseline (--seq2_del)", **metrics_baseline},
    {"Model": "Tuned (ngram4, seq2_max2, seq2_del, grow)", **metrics_tuned},
])
summary["PER (%)"] = (summary["per"] * 100).round(2)
summary["WER (%)"] = (summary["wer"] * 100).round(2)

summary[["Model", "PER (%)", "WER (%)", "n_words", "n_correct", "n_ref_phones", "n_edit"]]

,Model,PER (%),WER (%),n_words,n_correct,n_ref_phones,n_edit
0,Baseline (--seq2_del),7.40,37.24,2312,1451,18086,1339
1,"Tuned (ngram4, seq2_max2, seq2_del, grow)",8.13,41.09,2312,1362,18086,1470


## Error Analysis

Breaks down where the tuned model fails: by error type, phoneme unit, and substitution pair.
All analysis is on `hyp_tuned` vs `ref` (2 312 test words, 18 086 reference phones, 1 470 total edits).

In [ ]:
import numpy as np
from collections import Counter

err = collect_errors(ref, hyp_tuned)

total_errors = err['n_sub'] + err['n_del'] + err['n_ins']

error_div = pd.DataFrame([
    {"Type": "Substitution", "Count": err['n_sub'], "% of Errors": f"{err['n_sub']/total_errors*100:.1f}%"},
    {"Type": "Deletion",     "Count": err['n_del'], "% of Errors": f"{err['n_del']/total_errors*100:.1f}%"},
    {"Type": "Insertion",    "Count": err['n_ins'], "% of Errors": f"{err['n_ins']/total_errors*100:.1f}%"},
    {"Type": "TOTAL",        "Count": total_errors, "% of Errors": "100.0%"},
])
print(f"Total edit operations : {total_errors}")
error_div

In [ ]:
def _top10_with_stats(counter, label):
    counts = np.array(list(counter.values()))
    mean   = counts.mean()
    median = np.median(counts)
    print(f"
--- {label} ---")
    print(f"  Unique phones : {len(counts)}")
    print(f"  Mean count    : {mean:.2f}")
    print(f"  Median count  : {median:.1f}")
    return pd.DataFrame([{"Phone": p, "Count": c} for p, c in counter.most_common(10)])

print("=== Top-10 Phoneme Units per Error Type ===")

df_sub_ref = _top10_with_stats(err['sub_ref_counter'], "Substitutions — which REF phones are replaced")
df_sub_hyp = _top10_with_stats(err['sub_hyp_counter'], "Substitutions — what phones are predicted instead")
df_del     = _top10_with_stats(err['del_counter'],     "Deletions — which ref phones are dropped")
df_ins     = _top10_with_stats(err['ins_counter'],     "Insertions — which phones are added by the model")

print("
Ref phones most often substituted:")
display(df_sub_ref)
print("
Hyp phones most often wrongly predicted:")
display(df_sub_hyp)
print("
Most deleted ref phones:")
display(df_del)
print("
Most inserted hyp phones:")
display(df_ins)

In [ ]:
pair_rows = [
    {"Pair": f"{r} → {h}", "Ref": r, "Predicted": h, "Count": c}
    for (r, h), c in err['sub_pair_counter'].most_common(10)
]
df_pairs = pd.DataFrame(pair_rows)

pair_counts = np.array(list(err['sub_pair_counter'].values()))
print(f"Substitution pairs — mean: {pair_counts.mean():.2f}  |  median: {np.median(pair_counts):.1f}")
print()
df_pairs[["Pair", "Count"]]

In [ ]:
def find_outliers(counter, threshold_sd=1.0, jump_ratio=1.5):
    """Return list of (key, count, reason) for statistically notable items.

    Signal A: count >= mean + threshold_sd * std
    Signal B: ratio to the next-lower count >= jump_ratio (natural cluster break)
    """
    if not counter:
        return []
    counts = np.array(list(counter.values()), dtype=float)
    mean, std = counts.mean(), counts.std()
    stat_threshold = mean + threshold_sd * std

    sorted_items = counter.most_common()
    outliers = []
    for i, (key, count) in enumerate(sorted_items):
        reasons = []
        if count >= stat_threshold:
            reasons.append(f"count {count} >= mean+1σ ({stat_threshold:.1f})")
        if i + 1 < len(sorted_items):
            next_count = sorted_items[i + 1][1]
            if next_count > 0 and count / next_count >= jump_ratio:
                reasons.append(f"gap ×{count/next_count:.1f} to next")
        if reasons:
            outliers.append((key, count, "; ".join(reasons)))
    return outliers

sub_outliers  = find_outliers(err['sub_ref_counter'])
del_outliers  = find_outliers(err['del_counter'])
ins_outliers  = find_outliers(err['ins_counter'])
pair_outliers = find_outliers(err['sub_pair_counter'])

print("Notable substituted ref phones:", [(p, c) for p, c, _ in sub_outliers])
print("Notable deleted phones:        ", [(p, c) for p, c, _ in del_outliers])
print("Notable inserted phones:       ", [(p, c) for p, c, _ in ins_outliers])
print("Notable substitution pairs:    ", [(f'{r}→{h}', c) for (r,h), c, _ in pair_outliers])

In [ ]:
def _show_examples(key, examples_dict):
    examples = examples_dict.get(key, [])
    if not examples:
        print("    (no examples stored)")
        return
    for ex in examples:
        print(f"    {ex['word']:<22}  ref: [{ex['ref']}]  hyp: [{ex['hyp']}]  → {ex['op']}")

print("=" * 70)
print("NOTABLE FINDINGS — word examples")
print("=" * 70)

print("
>>> Substitutions (by ref phone) <<<")
for phone, count, reason in sub_outliers:
    print(f"  /{phone}/  count={count}  ({reason})")
    _show_examples(phone, err['sub_ref_examples'])

print("
>>> Deletions <<<")
for phone, count, reason in del_outliers:
    print(f"  /{phone}/  count={count}  ({reason})")
    _show_examples(phone, err['del_examples'])

print("
>>> Insertions <<<")
for phone, count, reason in ins_outliers:
    print(f"  /{phone}/  count={count}  ({reason})")
    _show_examples(phone, err['ins_examples'])

print("
>>> Substitution Pairs <<<")
for (r, h), count, reason in pair_outliers:
    print(f"  {r} → {h}  count={count}  ({reason})")
    _show_examples((r, h), err['sub_pair_examples'])